<a href="https://colab.research.google.com/github/tparlar/courses/blob/main/Intro_to_Exploratory_data_analysis_(EDA)_in_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
cooperunion_cardataset_path = kagglehub.dataset_download('CooperUnion/cardataset')

print('Data source import complete.')


# Python'da Keşifçi Veri Analizi
Exploratory Data Analysis (EDA)

## Python'da verileri nasıl keşfedeceğimizi anlayalım.

## Giriş

**Keşifçi Veri Analizi Nedir?**

Keşifçi Veri Analizi (KVA) veri setlerini ana özelliklerini anlamaktır. Bu adım, Makine Öğrenimi uygulamak için verileri modellerken çok önemlidir. Histogramlar, Kutu grafikleri, Dağılım grafikleri gibi görsellerden yararlanılır.

**Otomobil veri seti**

(https://www.kaggle.com/CooperUnion/cardataset) indirilebilir. Veri seti hakkında kısa bir bilgi vermek gerekirse, bu veri seti 10.000'den fazla satır ve Motor Yakıt Tipi, Motor Beygir Gücü (HP), Şanzıman Tipi, otoban MPG, şehir MPG gibi otomobil özelliklerini içeren 10'dan fazla sütun içermektedir. Bu eğitimde verileri keşfedecek ve modellemeye hazır hale getireceğiz.

---

## 1. KVA için gerekli kütüphaneleri içe aktarma

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns                       #Görselleştirme
import matplotlib.pyplot as plt             #Görselleştirme
%matplotlib inline
sns.set(color_codes=True)

---

## 2. Verileri veri çerçevesine yükleme.

Verileri pandas veri çerçevesine yüklemek, KVA'daki en önemli adımlardan biridir, çünkü veri setindeki değerlerin virgülle ayrıldığını görebiliriz. Yapmamız gereken tek şey CSV'yi bir veri çerçevesine okumak ve pandas veri çerçevesi bizim için bu işi yapar.

Veri setini not defterine almak veya yüklemek için tek bir basit adım attım. Google Colab'da not defterinin sol tarafında bir > (büyüktür sembolü) bulacaksınız. Buna tıkladığınızda üç seçenekli bir sekme göreceksiniz, sadece Dosyalar'ı seçmeniz gerekiyor. Ardından Yükle seçeneği yardımıyla dosyanızı kolayca yükleyebilirsiniz. Google Drive'a bağlamaya veya belirli kütüphaneler kullanmaya gerek yok, sadece veri setini yükleyin ve işiniz bitti. Bu adımda hatırlanması gereken bir şey, çalışma zamanı geri dönüştürüldüğünde yüklenen dosyaların silineceğidir. Veri setini not defterine bu şekilde aldım.

In [ ]:
df = pd.read_csv("../input/cardataset/data.csv")
# ilk 5 satir listelenir
df.head()

In [ ]:
df.tail()                        # son 5 satir listelenir

---

## 3. Veri türlerini kontrol etme

Burada veri tiplerini kontrol ediyoruz, çünkü bazen otomobilin MSRP'si veya fiyatı bir dize olarak saklanabilir; bu durumda, verileri bir grafik aracılığıyla çizebilmemiz için o dizeyi tam sayı verisine dönüştürmemiz gerekir. Burada, bu durumda, veriler zaten tam sayı biçiminde olduğundan endişelenecek bir şey yok.

In [ ]:
df.dtypes

---

## 4. İlgisiz sütunları silme

Bu adım her KVA'da kesinlikle gereklidir, çünkü bazen hiç kullanmadığımız birçok sütun olabilir, bu durumlarda bırakmak tek çözümdür. Bu durumda, Motor Yakıt Tipi, Piyasa Kategorisi, Araç Tarzı, Popülerlik, Kapı Sayısı, Araç Boyutu gibi sütunlar benim için hiçbir anlam ifade etmiyor, bu yüzden bu örnek için onları bıraktım.

In [ ]:
df = df.drop(['Engine Fuel Type', 'Market Category', 'Vehicle Style', 'Popularity', 'Number of Doors', 'Vehicle Size'], axis=1)
df.head(5)

---

## 5. Sütunları yeniden adlandırma

Bu örnekte, sütun adlarının çoğu okuması çok kafa karıştırıcıydı, bu yüzden sütun adlarını değiştirdim. Bu, veri setinin okunabilirliğini artıran iyi bir yaklaşımdır.

In [ ]:
df = df.rename(columns={"Engine HP": "HP", "Engine Cylinders": "Cylinders", "Transmission Type": "Transmission", "Driven_Wheels": "Drive Mode","highway MPG": "MPG-H", "city mpg": "MPG-C", "MSRP": "Price" })
df.head(5)

---

## 6. Yinelenen satırları bırakma

Bu genellikle kullanışlı bir şeydir, çünkü bu durumdaki gibi 10.000'den fazla satır içeren büyük bir veri seti, rahatsız edici olabilecek bazı yinelenen verilere sahip olabilir, bu yüzden burada veri setindeki tüm yinelenen değerleri kaldırıyorum. Örneğin, kaldırmadan önce 11914 satır verim vardı, ancak yinelenenleri kaldırdıktan sonra 10925 veri, yani 989 yinelenen verim vardı.

In [ ]:
df.shape

In [ ]:
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)

Şimdi yinelenen verileri kaldıralım çünkü onları kaldırmak sorun değil.

In [ ]:
df.count()      # Used to count the number of rows

Yukarıda görüldüğü gibi 11914 satır var ve 989 adet yinelenen satırı kaldırıyoruz.

In [ ]:
df = df.drop_duplicates()
df.head(5)

In [ ]:
df.count()

---

## 7. Eksik veya boş değerleri bırakma.

Bu, önceki adıma benzer, ancak burada tüm eksik değerler tespit edilir ve daha sonra bırakılır. Şimdi, bu iyi bir yaklaşım değildir, çünkü birçok kişi eksik değerleri o sütunun ortalaması veya medyanı ile değiştirir, ancak bu durumda ben sadece bu eksik değerleri bıraktım. Bunun nedeni, 10.000 değere kıyasla neredeyse 100 eksik değer olmasıdır, bu küçük bir sayıdır ve ihmal edilebilir, bu yüzden bu değerleri bıraktım.

In [ ]:
print(df.isnull().sum())

Yukarıdaki adımda hem Silindirler hem de Beygir Gücü (HP) sayılırken 10925 satır üzerinden 10856 ve 10895'e sahip olmasının nedeni budur.

In [ ]:
df = df.dropna()    # Dropping the missing values.
df.count()

Şimdi boş veya N/A değerleri içeren tüm satırları kaldırdık (Silindirler ve Beygir Gücü (HP)).

In [ ]:
print(df.isnull().sum())   # After dropping the values

---

## 8. Aykırı değerleri tespit etme

Aykırı değer, diğer noktalardan farklı olan bir nokta veya bir dizi noktadır. Bazen çok yüksek veya çok düşük olabilirler. Aykırı değerleri tespit etmek ve kaldırmak genellikle iyi bir fikirdir. Çünkü aykırı değerler, daha az doğru bir modelle sonuçlanmanın birincil nedenlerinden biridir. Bu nedenle, onları kaldırmak iyi bir fikirdir. Gerçekleştireceğim aykırı değer tespiti ve kaldırma, IQR puan tekniği olarak adlandırılır. Aykırı değerler genellikle bir kutu grafiği kullanılarak görselleştirmelerle görülebilir. Aşağıda MSRP, Silindirler, Beygir Gücü ve Motor Boyutu'nun kutu grafiği gösterilmiştir. Tüm grafiklerde, kutunun dışında kalan bazı noktalar bulabilirsiniz, bunlar aykırı değerlerden başka bir şey değildir. Bu atamada gerçekleştirdiğim aykırı değer bulma ve kaldırma tekniği, [towards data science](https://towardsdatascience.com/ways-to-detect-and-remove-the-outliers-404d16608dba) adlı bir eğitimden yardım alınarak yapılmıştır.

In [ ]:
sns.boxplot(x=df['Price'])

In [ ]:
sns.boxplot(x=df['HP'])

In [ ]:
sns.boxplot(x=df['Cylinders'])

In [ ]:
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
print(IQR)

Yukarıdaki değerler hakkında endişelenmeyin, çünkü her birini bilmek önemli değildir, sadece aykırı değerleri kaldırmak için bu tekniği nasıl kullanacağınızı bilmek önemlidir.

In [ ]:
df = df[~((df < (Q1 - 1.5 * IQR)) |(df > (Q3 + 1.5 * IQR))).any(axis=1)]
df.shape

Yukarıda görüldüğü gibi yaklaşık 1600 satır aykırı değerdi. Ancak aykırı değerleri tamamen kaldıramazsınız, çünkü yukarıdaki tekniği kullandıktan sonra bile 1-2 aykırı değer kalabilir ama bu sorun değil çünkü 100'den fazla aykırı değer vardı. Hiçbir şeyden iyidir.

---

## 9. Farklı özellikleri birbirine karşı (dağılım), frekansa karşı (histogram) çizme

### Histogram

Histogram, bir aralıktaki değişkenlerin oluşum sıklığını ifade eder. Bu durumda, başlıca 10 farklı otomobil üretim şirketi bulunmaktadır, ancak en çok arabaya kimin sahip olduğunu bilmek genellikle önemlidir. Bunu yapmak için histogram, farklı bir şirket tarafından üretilen toplam araba sayısını bilmemizi sağlayan basit çözümlerden biridir.

In [ ]:
df.Make.value_counts().nlargest(40).plot(kind='bar', figsize=(10,5))
plt.title("Number of cars by make")
plt.ylabel('Number of cars')
plt.xlabel('Make');

### Isı Haritaları

Isı Haritaları, bağımlı değişkenleri bulmamız gerektiğinde gerekli olan bir grafik türüdür. Özellikler arasındaki ilişkiyi bulmanın en iyi yollarından biri ısı haritaları kullanmaktır. Aşağıdaki ısı haritasında, fiyat özelliğinin esas olarak Motor Boyutu, Beygir Gücü ve Silindirler'e bağlı olduğunu biliyoruz.

In [ ]:
plt.figure(figsize=(10,5))
c= df.corr()
sns.heatmap(c,cmap="BrBG",annot=True)
c

### Dağılım Grafiği

Genellikle iki değişken arasındaki korelasyonu bulmak için dağılım grafikleri kullanırız. Burada dağılım grafikleri Beygir Gücü ve Fiyat arasında çizilmiştir ve aşağıdaki grafiği görebiliriz. Aşağıda verilen grafikle kolayca bir eğilim çizgisi çizebiliriz. Bu özellikler iyi bir noktalar saçılımı sağlar.

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
ax.scatter(df['HP'], df['Price'])
ax.set_xlabel('HP')
ax.set_ylabel('Price')
plt.show()

**Dolayısıyla, yukarıdakiler Keşifçi veri analizinde yer alan adımlardan bazılarıdır, bunlar iyi bir KVA yapmak için takip etmeniz gereken bazı genel adımlardır. Daha gelecek çok şey var, ancak şimdilik, herhangi bir veri seti verildiğinde iyi bir KVA'nın nasıl yapılacağına dair yeterince fikir veriyor. Daha fazla güncelleme için bizi takipte kalın.**

## Teşekkürler.